# GEDI Footprint Extraction Workflow

Extracts GEDI Level 2A shots over an area of interest, filters to high-quality returns, and exports point and footprint (25 m buffer) GeoJSON files.

**Bounding box:** Set from a `.gdb` boundary file (interactive dialog) **or** entered manually below.

---
## Setup

Install required packages if you haven't already:

```bash
# in Google Colab:
#   !pip install earthaccess h5py numpy pandas geopandas shapely fiona -q
# in VS Code terminal:
#   python -m pip install earthaccess h5py numpy pandas geopandas shapely fiona
```

In [1]:
#------------ imports ----------------------------------------------------------
import earthaccess
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import datetime
import os
import tkinter as tk
from tkinter import filedialog, simpledialog
import fiona

c:\Users\davisk10\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Step 1: Login to NASA Earthdata

In [2]:
# ---------- Login to NASA Earthdata -------------------------------------------
# prompts for NASA Earthdata username and password
earthaccess.login()

---
## Step 2: Set Output Folder and File Header

A dialog will open asking you to:
1. Select the folder where output files (CSV, GeoJSON) will be saved
2. Enter a short header used to name all output files (e.g. `bpines` → `bpines_gedi_shots.csv`)

In [2]:
print("Opening dialogs — check your taskbar if they don't appear on top.\n")

# ── helper: build a fresh hidden root each time ──────────────────────────────
def make_root():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    root.update()
    return root

# ── 1. Output folder ─────────────────────────────────────────────────────────
root = make_root()
data_folder = filedialog.askdirectory(
    title="Select your output folder (where files will be saved)",
    parent=root
)
root.destroy()

if not data_folder:
    raise SystemExit("No output folder selected. Exiting.")
print(f"Output folder: {data_folder}")

# ── 2. File name header ───────────────────────────────────────────────────────
root = make_root()
file_header = simpledialog.askstring(
    "File name header",
    "Enter a short header for all output file names\n(e.g. 'bpines' → bpines_gedi_shots.csv):",
    parent=root
)
root.destroy()

if not file_header:
    raise SystemExit("No file header entered. Exiting.")
file_header = file_header.strip().lower().replace(" ", "_")
print(f"File header: {file_header}")

# ── 3. Build output paths ─────────────────────────────────────────────────────
csv_path            = os.path.join(data_folder, f'{file_header}_gedi_shots.csv')
geojson_path        = os.path.join(data_folder, f'{file_header}_gedi_shots.geojson')
buffer_geojson_path = os.path.join(data_folder, f'{file_header}_gedi_footprints.geojson')

print(f"\nOutput files will be:")
print(f"  {csv_path}")
print(f"  {geojson_path}")
print(f"  {buffer_geojson_path}")

Opening dialogs — check your taskbar if they don't appear on top.

Output folder: C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files
File header: arb_test1

Output files will be:
  C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.csv
  C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test1_gedi_shots.geojson
  C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/CODE/Notebook_Test1_Arb_Output_Files\arb_test1_gedi_footprints.geojson


---
## Step 3: Define Area of Interest Bounding Box

**Option A** — Load from a `.gdb` boundary file (recommended): run the cell below.

**Option B** — Enter coordinates manually: skip to the manual cell further below.

### Option A: Load boundary from .gdb

In [ ]:
from tkinter import filedialog, Tk

# Required to prevent a blank window from appearing
root = Tk()
root.withdraw()

gdb_path = filedialog.askdirectory(title="Select your .gdb folder (boundary for ROI)")

if not gdb_path:
    raise SystemExit("No .gdb folder selected. Exiting.")



# Select the boundary GDB
gdb_path = filedialog.askdirectory(title="Select your .gdb folder (boundary for ROI)")

if not gdb_path:
    raise SystemExit("No .gdb folder selected. Exiting.")

# List available layers
layers = fiona.listlayers(gdb_path)
print("\nLayers found in geodatabase:")
for i, layer in enumerate(layers):
    print(f"  {i + 1}: {layer}")

# Ask user to pick a layer
layer_index = simpledialog.askinteger(
    "Select Layer",
    f"Enter the number of the layer to use (1–{len(layers)}):",
    minvalue=1,
    maxvalue=len(layers)
)

# All dialogs done — destroy root cleanly
root.destroy()

if layer_index is None:
    raise SystemExit("No layer selected. Exiting.")

layer_name = layers[layer_index - 1]
print(f"\nUsing layer: {layer_name}")

# Load the boundary layer and compute bounding box
boundary = gpd.read_file(gdb_path, layer=layer_name)
boundary_wgs84 = boundary.to_crs("EPSG:4326")  # Ensure WGS84 lat/lon

minx, miny, maxx, maxy = boundary_wgs84.total_bounds
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = minx, miny, maxx, maxy

print(f"\nBounding box derived from .gdb boundary:")
print(f"LAT_MIN, LAT_MAX = {LAT_MIN:.6f}, {LAT_MAX:.6f}")
print(f"LON_MIN, LON_MAX = {LON_MIN:.6f}, {LON_MAX:.6f}")

### Option B: Insert coordinates manually

Only run this cell if you skipped Option A.

In [ ]:
# -------- Insert coordinates for area of interest -----------------------------
LAT_MIN, LAT_MAX = 35.309, 35.312
LON_MIN, LON_MAX = -120.664, -120.657

# Destroy root here since Option A (GDB cell) was skipped
try:
    root.destroy()
except:
    pass

---
## Step 4: Search for GEDI Granules

In [ ]:
# ---------- Search for GEDI granules over area of interest --------------------
print('Searching for GEDI granules...')
results = earthaccess.search_data(
    short_name='GEDI02_A',
    version='002',
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
)
print(f'Found {len(results)} granules covering your arboretum')

---
## Step 5: Stream Granules and Extract Shots

Streams each granule directly — no download needed. (~20 min depending on granule count)

In [ ]:
# --- Stream each granule and extract shots (~20 min) --------------------------
all_shots = []
for granule in results:
    filename = granule['meta']['native-id']
    print(f'\nProcessing: {filename}')

    try:
        # Stream the file directly — no download needed
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                # Filter to arboretum bounding box
                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                print(f'  {beam}: {mask.sum()} shots in arboretum')

                rh = h5file[beam]['rh'][mask]

                # Date from filename
                try:
                    year = int(filename[9:13])
                    doy  = int(filename[13:16])
                    date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
                    date_str = date.strftime('%Y-%m-%d')
                except:
                    date_str = 'unknown'

                shot_data = {
                    'filename':     filename,
                    'beam':         beam,
                    'date':         date_str,
                    'lat':          lat[mask],
                    'lon':          lon[mask],
                    'rh25':         rh[:, 25],
                    'rh50':         rh[:, 50],
                    'rh75':         rh[:, 75],
                    'rh75':         rh[:, 85],
                    'rh95':         rh[:, 95],  
                    'rh98':         rh[:, 98], #relative height at 98th percentile (top of forest canopy height in m)
                    'rh99':         rh[:, 99],
                    'rh100':        rh[:, 100],
                    'sensitivity':  h5file[beam]['sensitivity'][mask], #how well detected ground return through the canopy, ranges from 0 to 1
                                                                        #0.9 - 1.0 → excellent — laser punched through canopy cleanly and found ground
                    'quality_flag': h5file[beam]['quality_flag'][mask],
                    'degrade_flag': h5file[beam]['degrade_flag'][mask],
                    'shot_number':  h5file[beam]['shot_number'][mask].astype(str),
                }

                all_shots.append(pd.DataFrame(shot_data))

            except KeyError as e:
                print(f'  Skipping {beam} — missing field: {e}')
                continue

        h5file.close()

    except Exception as e:
        print(f'  Error: {e}')
        continue

---
## Step 6: Filter to High-Quality Shots and Save Outputs

In [ ]:
# ------------- Combine all shots ----------------------------------------------
if len(all_shots) == 0:
    print('\nNo shots found in arboretum area')
else:
    combined = pd.concat(all_shots, ignore_index=True)
    print(f'\nTotal shots found: {len(combined)}')

    # Filter to high quality shots only
    quality = combined[
        (combined['quality_flag'] == 1) &
        (combined['degrade_flag'] == 0)
    ].copy()
    print(f'High quality shots: {len(quality)}')
    print(quality[['date', 'beam', 'lat', 'lon', 'rh98', 'sensitivity']].to_string())

    # Ensure shot_number stays as string (ArcGIS Pro compatibility)
    quality['shot_number'] = quality['shot_number'].astype(str)

    # Save CSV
    quality.to_csv(csv_path, index=False)
    print(f'\nCSV saved: {csv_path}')

---
## Step 7: Export Point GeoJSON

In [ ]:
# ── Step 5: Build point GeoDataFrame ──────────────────────────────────────────
if len(quality) == 0:
    print('No quality shots to export. Exiting.')
else:
    geometry_pts = [Point(lon, lat) for lon, lat in zip(quality['lon'], quality['lat'])]
    gdf_points = gpd.GeoDataFrame(quality, geometry=geometry_pts, crs='EPSG:4326')

    # Save point GeoJSON
    gdf_points.to_file(geojson_path, driver='GeoJSON')
    print(f'Point GeoJSON saved: {geojson_path}')

---
## Step 8: Create 25 m Footprint Buffers and Export

**WHY WE REPROJECT:**  
GEDI points are stored in WGS84 (EPSG:4326), where coordinates are in degrees. Buffering in degrees does NOT give you metres — 0.0001° of longitude is ~9 m near the equator but changes with latitude.

Solution: reproject to a CRS measured in metres, buffer by exactly 12.5 m (radius), then reproject back to WGS84.

UTM Zone 10N (EPSG:32610) covers central California and is appropriate for this study area. Adjust the EPSG code if your ROI is in a different UTM zone.

**WHAT THE BUFFER DOES:**  
Each GEDI shot records a single lat/lon centre point, but the real laser footprint illuminates a ~25 m diameter circle on the ground. `buffer(12.5)` expands each point into a circle with radius 12.5 m, giving a 25 m diameter polygon that matches the physical footprint. When you later extract biomass, canopy height, or image pixel values WITHIN these polygons, you are sampling the same area the lidar saw.

In [ ]:
# ── Step 6: Create 25 m footprint buffers ─────────────────────────────────────
UTM_CRS = 'EPSG:32610'   # UTM Zone 10N — change if your site is elsewhere

gdf_utm = gdf_points.to_crs(UTM_CRS)
gdf_utm['geometry'] = gdf_utm.geometry.buffer(12.5)    # 12.5 m radius → 25 m diameter
gdf_footprints = gdf_utm.to_crs('EPSG:4326')           # reproject back to WGS84

gdf_footprints.to_file(buffer_geojson_path, driver='GeoJSON')
print(f'Footprint GeoJSON saved (25 m buffers): {buffer_geojson_path}')

print('\nDone! Load the GeoJSON files into QGIS or ArcGIS Pro.')
print('Outputs:')
print(f'  Points     → {geojson_path}')
print(f'  Footprints → {buffer_geojson_path}')
print(f'  CSV        → {csv_path}')

---
## Diagnostic: Why Were Shots Rejected?

Re-streams all granules without quality filtering and prints a breakdown of rejection reasons. Run this if you want to understand why shots were excluded.

In [ ]:
# ------ Diagnostic: why were shots rejected? ----------------------------------

all_shots_unfiltered = []

for granule in results:
    filename = granule['meta']['native-id']

    try:
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                quality_flag = h5file[beam]['quality_flag'][mask]
                degrade_flag = h5file[beam]['degrade_flag'][mask]
                sensitivity  = h5file[beam]['sensitivity'][mask]

                # Date from filename
                try:
                    year = int(filename[9:13])
                    doy  = int(filename[13:16])
                    date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
                    date_str = date.strftime('%Y-%m-%d')
                except:
                    date_str = 'unknown'

                for i in range(mask.sum()):
                    all_shots_unfiltered.append({
                        'filename':     filename,
                        'beam':         beam,
                        'date':         date_str,
                        'lat':          lat[mask][i],
                        'lon':          lon[mask][i],
                        'quality_flag': quality_flag[i],
                        'degrade_flag': degrade_flag[i],
                        'sensitivity':  sensitivity[i],
                        'rejected':     quality_flag[i] != 1 or degrade_flag[i] != 0
                    })

            except KeyError as e:
                continue

        h5file.close()

    except Exception as e:
        print(f'Error: {e}')
        continue

# ── Summary ───────────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_shots_unfiltered)

print(f'Total shots in bounding box: {len(df_all)}')
print(f'Accepted (quality=1, degrade=0): {len(df_all[~df_all["rejected"]])}')
print(f'Rejected: {len(df_all[df_all["rejected"]])}')
print()
print('Rejection breakdown:')
print(f'  quality_flag = 0: {len(df_all[df_all["quality_flag"] == 0])}')
print(f'  degrade_flag != 0: {len(df_all[df_all["degrade_flag"] != 0])}')
print()
print('Rejected shots by date and beam:')
rejected = df_all[df_all['rejected']]
print(rejected[['date', 'beam', 'lat', 'lon', 'quality_flag', 'degrade_flag', 'sensitivity']].to_string())

At this point in the code, (with the hope that i am somehow able to connect the R code for creating the CHM for the study area before), I should have:
    1. the CHM for the study area
    2. the GEDI footprint locations 
    3. the GEDI height metrics for each footprint
    4. the buffered 12.5m radius rings for each footprint

What I will need to do next in this code is the following: 
1. incoroporate the Monte Carlo algorithm (I think it is better than the UAV paper method because the randomization is better i think?)


Monte Carlo Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
==========================================================================================
Based on:
  - 'Simulation-Based Correction of Geolocation Errors in GEDI Footprint Positions
     Using Monte Carlo Approach'
  - 'The impact of geolocation uncertainty on GEDI tropical forest canopy height
     estimation and change monitoring'

Method:
  Each GEDI footprint location is shifted with randomly generated position errors
  modelled using the GEDI geolocation uncertainty (Dubayah et al., 2020a):

      x*_i = x + s_i * cos(theta_i)
      y*_i = y + s_i * sin(theta_i)

  where:
    (x*_i, y*_i) = shifted GEDI footprint centre coordinate
    (x, y)       = GEDI product reported footprint centre coordinate
    s_i          ~ N(mu=0 m, sigma=10 m)  [geolocation uncertainty]
    theta_i      ~ Uniform(0, 2*pi)       [random direction]
    n            = 300 simulations per footprint